# AWS Kinesis — Data Streams, Firehose, Analytics

## Mental Model

Kinesis is AWS-native streaming infrastructure for durable ingestion, fan-out consumption, and low-ops delivery into the data lake.

- **Kinesis Data Streams** = ordered stream shards that producers write to and consumers read from
- **Kinesis Firehose** = managed delivery pipeline into S3 and analytics destinations
- **Kinesis Analytics patterns** = near-real-time transforms, aggregations, and downstream lakehouse queries

### Citi framing

Citi operates **6,000+ API endpoints** where latency, throughput, and error-rate telemetry produce continuous event streams.  
A common AWS-native pattern is:

`Producers → Kinesis Data Streams / Firehose → S3 → Athena / downstream analytics`

In this notebook we will:
1. Connect to AWS using the **study** profile in **us-east-1**
2. Pull **50 alert records** from PostgreSQL
3. Send them into **Kinesis Data Streams**
4. Read them back from a shard
5. Send the same records into **Kinesis Firehose**
6. Confirm S3 delivery
7. Size shard requirements for a Citi-scale scenario
8. Compare **Kinesis vs Kafka** in decision-table form


In [ ]:
import json
import time
import uuid
import random
from datetime import datetime, timezone
from decimal import Decimal
from typing import List, Dict, Any

import boto3
import psycopg2
from botocore.exceptions import ClientError, WaiterError

AWS_PROFILE = "study"
AWS_REGION = "us-east-1"
AWS_ACCOUNT_ID = "357811130281"

PG_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "dbname": "de_telemetry",
    "user": "de_admin",
    "password": "DeAdmin2026!"
}

KINESIS_STREAM_NAME = "citi-telemetry-stream"
FIREHOSE_STREAM_NAME = f"citi-firehose-{uuid.uuid4().hex[:10]}"
ROLE_NAME = f"firehose_delivery_role_{uuid.uuid4().hex[:8]}"
ROLE_POLICY_NAME = "firehose_s3_delivery_policy"

print("Configuration loaded.")
print(f"AWS profile={AWS_PROFILE} region={AWS_REGION} account={AWS_ACCOUNT_ID}")
print(f"Kinesis stream={KINESIS_STREAM_NAME}")
print(f"Firehose delivery stream={FIREHOSE_STREAM_NAME}")


## Setup

This notebook uses:
- **boto3** with AWS profile **study**
- **PostgreSQL** for sample telemetry alerts
- AWS clients for:
  - **Kinesis Data Streams**
  - **Firehose**
  - **S3**
  - **IAM**
  - **STS**

No package-install cells are included because the environment is assumed pre-provisioned.


In [ ]:
session = boto3.Session(profile_name=AWS_PROFILE, region_name=AWS_REGION)

sts = session.client("sts")
kinesis = session.client("kinesis")
firehose = session.client("firehose")
s3 = session.client("s3")
iam = session.client("iam")

identity = sts.get_caller_identity()
print("Authenticated AWS identity:")
print(json.dumps(identity, indent=2))


In [ ]:
def get_alert_records(limit: int = 50) -> List[Dict[str, Any]]:
    sql = '''
        SELECT
            a.alert_id,
            a.endpoint_id,
            a.severity,
            a.message,
            a.created_at,
            e.name AS endpoint_name,
            e.region,
            e.status,
            e.category
        FROM alerts a
        JOIN endpoints e
          ON a.endpoint_id = e.endpoint_id
        ORDER BY a.created_at DESC
        LIMIT %s
    '''
    with psycopg2.connect(**PG_CONFIG) as conn:
        with conn.cursor() as cur:
            cur.execute(sql, (limit,))
            rows = cur.fetchall()

    records = []
    for row in rows:
        alert_id, endpoint_id, severity, message, created_at, endpoint_name, region, status, category = row
        records.append({
            "alert_id": alert_id,
            "endpoint_id": endpoint_id,
            "severity": severity,
            "message": message,
            "created_at": created_at.astimezone(timezone.utc).isoformat(),
            "endpoint_name": endpoint_name,
            "region": region,
            "status": status,
            "category": category,
            "source": "postgres.alerts",
            "narrative": "Citi monitors 6000+ API endpoints for latency, error rate, and throughput."
        })
    return records

alert_records = get_alert_records(50)
print(f"Loaded {len(alert_records)} alert records from PostgreSQL.")
print("First record preview:")
print(json.dumps(alert_records[0], indent=2))


## Kinesis Data Streams

We will:
1. Create **citi-telemetry-stream** with **1 shard**
2. Push **50 records** from the alerts table
3. Discover the shard
4. Read records back using a shard iterator
5. Print the first 5 records
6. Delete the stream

### Shard capacity math

For standard Kinesis Data Streams shard sizing:
- **Write**: up to **1 MB/sec** or **1,000 records/sec** per shard
- **Read**: up to **2 MB/sec** per shard

So a single shard is perfect for this demo, but not for Citi-scale production throughput.


In [ ]:
def ensure_stream_deleted(stream_name: str) -> None:
    try:
        kinesis.delete_stream(StreamName=stream_name, EnforceConsumerDeletion=True)
    except ClientError as e:
        code = e.response["Error"]["Code"]
        if code not in {"ResourceNotFoundException"}:
            raise

def wait_for_stream_absent(stream_name: str, timeout_seconds: int = 180) -> None:
    start = time.time()
    while time.time() - start < timeout_seconds:
        try:
            kinesis.describe_stream_summary(StreamName=stream_name)
            time.sleep(3)
        except ClientError as e:
            if e.response["Error"]["Code"] == "ResourceNotFoundException":
                return
            raise
    raise TimeoutError(f"Timed out waiting for stream deletion: {stream_name}")

def create_kinesis_stream(stream_name: str, shard_count: int = 1) -> None:
    try:
        kinesis.describe_stream_summary(StreamName=stream_name)
        print(f"Stream {stream_name} already exists. Reusing it.")
        waiter = kinesis.get_waiter("stream_exists")
        waiter.wait(StreamName=stream_name, WaiterConfig={"Delay": 3, "MaxAttempts": 40})
        return
    except ClientError as e:
        if e.response["Error"]["Code"] != "ResourceNotFoundException":
            raise

    kinesis.create_stream(StreamName=stream_name, ShardCount=shard_count)
    waiter = kinesis.get_waiter("stream_exists")
    waiter.wait(StreamName=stream_name, WaiterConfig={"Delay": 3, "MaxAttempts": 40})
    print(f"Created stream {stream_name} with {shard_count} shard(s).")

def put_alerts_to_kinesis(stream_name: str, records: List[Dict[str, Any]]) -> Dict[str, Any]:
    entries = []
    for record in records:
        entries.append({
            "Data": (json.dumps(record) + "\n").encode("utf-8"),
            "PartitionKey": str(record["endpoint_id"])
        })
    response = kinesis.put_records(StreamName=stream_name, Records=entries)
    return response

def get_first_shard_id(stream_name: str) -> str:
    desc = kinesis.describe_stream(StreamName=stream_name)
    return desc["StreamDescription"]["Shards"][0]["ShardId"]

def read_records_from_shard(stream_name: str, shard_id: str, max_records: int = 50, max_polls: int = 10) -> List[Dict[str, Any]]:
    iterator_resp = kinesis.get_shard_iterator(
        StreamName=stream_name,
        ShardId=shard_id,
        ShardIteratorType="TRIM_HORIZON"
    )
    shard_iterator = iterator_resp["ShardIterator"]

    collected = []
    polls = 0
    while shard_iterator and len(collected) < max_records and polls < max_polls:
        resp = kinesis.get_records(ShardIterator=shard_iterator, Limit=min(100, max_records))
        shard_iterator = resp.get("NextShardIterator")
        for item in resp.get("Records", []):
            payload = item["Data"].decode("utf-8").strip()
            if payload:
                collected.append(json.loads(payload))
                if len(collected) >= max_records:
                    break
        polls += 1
        if len(collected) < max_records:
            time.sleep(1)

    return collected

create_kinesis_stream(KINESIS_STREAM_NAME, shard_count=1)
put_response = put_alerts_to_kinesis(KINESIS_STREAM_NAME, alert_records)

print("PutRecords response summary:")
print(json.dumps({
    "FailedRecordCount": put_response.get("FailedRecordCount"),
    "RecordCount": len(put_response.get("Records", []))
}, indent=2))

time.sleep(3)

shard_id = get_first_shard_id(KINESIS_STREAM_NAME)
print(f"Shard discovered: {shard_id}")

replayed_records = read_records_from_shard(KINESIS_STREAM_NAME, shard_id, max_records=50, max_polls=12)
print(f"Read back {len(replayed_records)} records from shard.")

print("\nFirst 5 records:")
for idx, record in enumerate(replayed_records[:5], start=1):
    print(f"--- Record {idx} ---")
    print(json.dumps(record, indent=2))

ensure_stream_deleted(KINESIS_STREAM_NAME)
wait_for_stream_absent(KINESIS_STREAM_NAME)
print(f"Deleted stream: {KINESIS_STREAM_NAME}")


## Kinesis Firehose

We will:
1. Create or reuse an S3 bucket
2. Create an IAM role that Firehose can assume
3. Create a Firehose delivery stream targeting S3
4. Put **50 records**
5. Wait up to **60 seconds** for buffering and delivery
6. Confirm objects in S3
7. Delete the delivery stream

### Firehose buffering

Firehose delivers to S3 based on buffering conditions:
- **1 MB** payload size, or
- **60 seconds** buffer interval,
- whichever happens first

That is why Firehose is ideal for **zero-ops ingestion into the data lake**, but not for ultra-low-latency per-record consumption.


In [ ]:
def ensure_bucket(bucket_name: str) -> str:
    try:
        s3.head_bucket(Bucket=bucket_name)
        print(f"Using existing bucket: {bucket_name}")
        return bucket_name
    except ClientError:
        pass

    candidate = f"{bucket_name}-{uuid.uuid4().hex[:8]}"
    if AWS_REGION == "us-east-1":
        s3.create_bucket(Bucket=candidate)
    else:
        s3.create_bucket(
            Bucket=candidate,
            CreateBucketConfiguration={"LocationConstraint": AWS_REGION}
        )
    print(f"Created bucket: {candidate}")
    return candidate

def create_firehose_role(bucket_name: str) -> str:
    trust_policy = {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Effect": "Allow",
                "Principal": {"Service": "firehose.amazonaws.com"},
                "Action": "sts:AssumeRole"
            }
        ]
    }

    role_arn = None
    try:
        response = iam.create_role(
            RoleName=ROLE_NAME,
            AssumeRolePolicyDocument=json.dumps(trust_policy),
            Description="Temporary Firehose delivery role for notebook demo"
        )
        role_arn = response["Role"]["Arn"]
        print(f"Created IAM role: {ROLE_NAME}")
    except ClientError as e:
        if e.response["Error"]["Code"] == "EntityAlreadyExists":
            role_arn = iam.get_role(RoleName=ROLE_NAME)["Role"]["Arn"]
            print(f"Reusing IAM role: {ROLE_NAME}")
        else:
            raise

    bucket_arn = f"arn:aws:s3:::{bucket_name}"
    policy_doc = {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Effect": "Allow",
                "Action": [
                    "s3:AbortMultipartUpload",
                    "s3:GetBucketLocation",
                    "s3:GetObject",
                    "s3:ListBucket",
                    "s3:ListBucketMultipartUploads",
                    "s3:PutObject"
                ],
                "Resource": [
                    bucket_arn,
                    f"{bucket_arn}/*"
                ]
            },
            {
                "Effect": "Allow",
                "Action": [
                    "logs:PutLogEvents"
                ],
                "Resource": "*"
            }
        ]
    }

    iam.put_role_policy(
        RoleName=ROLE_NAME,
        PolicyName=ROLE_POLICY_NAME,
        PolicyDocument=json.dumps(policy_doc)
    )

    time.sleep(10)
    return role_arn

def create_firehose_delivery_stream(stream_name: str, bucket_name: str, role_arn: str, prefix: str = "kinesis_firehose/") -> None:
    try:
        firehose.describe_delivery_stream(DeliveryStreamName=stream_name)
        print(f"Firehose stream {stream_name} already exists. Reusing it.")
        return
    except ClientError as e:
        if e.response["Error"]["Code"] != "ResourceNotFoundException":
            raise

    firehose.create_delivery_stream(
        DeliveryStreamName=stream_name,
        DeliveryStreamType="DirectPut",
        ExtendedS3DestinationConfiguration={
            "RoleARN": role_arn,
            "BucketARN": f"arn:aws:s3:::{bucket_name}",
            "Prefix": prefix,
            "ErrorOutputPrefix": "kinesis_firehose_errors/",
            "BufferingHints": {
                "SizeInMBs": 1,
                "IntervalInSeconds": 60
            },
            "CompressionFormat": "UNCOMPRESSED"
        }
    )
    print(f"Creating Firehose delivery stream: {stream_name}")

    start = time.time()
    while time.time() - start < 300:
        desc = firehose.describe_delivery_stream(DeliveryStreamName=stream_name)
        status = desc["DeliveryStreamDescription"]["DeliveryStreamStatus"]
        print(f"Firehose status: {status}")
        if status == "ACTIVE":
            return
        time.sleep(5)

    raise TimeoutError(f"Timed out waiting for Firehose stream to become ACTIVE: {stream_name}")

def put_records_to_firehose(stream_name: str, records: List[Dict[str, Any]]) -> None:
    batch = [{"Data": (json.dumps(r) + "\n").encode("utf-8")} for r in records]
    response = firehose.put_record_batch(DeliveryStreamName=stream_name, Records=batch)
    failed = response.get("FailedPutCount", 0)
    print(json.dumps({
        "RequestedRecords": len(records),
        "FailedPutCount": failed
    }, indent=2))
    if failed:
        raise RuntimeError("One or more Firehose records failed to put.")

def wait_for_s3_delivery(bucket_name: str, prefix: str, timeout_seconds: int = 90) -> List[Dict[str, Any]]:
    start = time.time()
    while time.time() - start < timeout_seconds:
        resp = s3.list_objects_v2(Bucket=bucket_name, Prefix=prefix)
        contents = resp.get("Contents", [])
        if contents:
            return contents
        time.sleep(5)
    return []

def delete_firehose_stream(stream_name: str) -> None:
    try:
        firehose.delete_delivery_stream(
            DeliveryStreamName=stream_name,
            AllowForceDelete=True
        )
        print(f"Delete initiated for Firehose delivery stream: {stream_name}")
    except ClientError as e:
        if e.response["Error"]["Code"] != "ResourceNotFoundException":
            raise

firehose_bucket = ensure_bucket("egirgis-lab")
firehose_role_arn = create_firehose_role(firehose_bucket)
firehose_prefix = f"kinesis_firehose/{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}/"

create_firehose_delivery_stream(
    FIREHOSE_STREAM_NAME,
    firehose_bucket,
    firehose_role_arn,
    prefix=firehose_prefix
)

put_records_to_firehose(FIREHOSE_STREAM_NAME, alert_records)

print("Waiting up to 60+ seconds for Firehose buffering and S3 delivery...")
delivered_objects = wait_for_s3_delivery(firehose_bucket, firehose_prefix, timeout_seconds=95)

print(f"Delivered object count: {len(delivered_objects)}")
for obj in delivered_objects[:10]:
    print(json.dumps({
        "Key": obj["Key"],
        "Size": obj["Size"],
        "LastModified": obj["LastModified"].astimezone(timezone.utc).isoformat()
    }, indent=2))

delete_firehose_stream(FIREHOSE_STREAM_NAME)


## Shard Calculator

Citi scenario:

- **6,000 endpoints**
- **10 events/sec each**
- **60,000 events/sec total**
- **500 bytes/event**
- Total write throughput = **30,000,000 bytes/sec ≈ 30 MB/sec**

Since one shard supports roughly **1 MB/sec writes**, the write-throughput minimum is **30 shards**.


In [ ]:
import math

def kinesis_shard_count(events_per_sec: int, avg_bytes_per_event: int) -> int:
    bytes_per_sec = events_per_sec * avg_bytes_per_event
    mb_per_sec = bytes_per_sec / 1_000_000
    shards_by_throughput = math.ceil(mb_per_sec / 1.0)  # 1 MB/s write per shard
    shards_by_record_rate = math.ceil(events_per_sec / 1000)  # 1000 records/s per shard
    return max(shards_by_throughput, shards_by_record_rate)

events_per_sec = 6000 * 10
avg_bytes_per_event = 500
required_shards = kinesis_shard_count(events_per_sec, avg_bytes_per_event)

print(json.dumps({
    "events_per_sec": events_per_sec,
    "avg_bytes_per_event": avg_bytes_per_event,
    "bytes_per_sec": events_per_sec * avg_bytes_per_event,
    "estimated_mb_per_sec": (events_per_sec * avg_bytes_per_event) / 1_000_000,
    "required_shards": required_shards
}, indent=2))


## Kinesis vs Kafka Decision Table

In [ ]:
decision_rows = [
    {
        "dimension": "ordering",
        "kinesis": "Guaranteed within a shard",
        "kafka": "Guaranteed within a partition",
        "when_to_use": "Both are strong if key-based ordering matters"
    },
    {
        "dimension": "replay",
        "kinesis": "Retention-based replay, simpler operational model",
        "kafka": "Excellent replay and long-lived log semantics",
        "when_to_use": "Kafka wins when replay/history is central"
    },
    {
        "dimension": "managed_overhead",
        "kinesis": "Very low in AWS",
        "kafka": "Higher unless using MSK/Confluent",
        "when_to_use": "Kinesis for AWS-native low-ops teams"
    },
    {
        "dimension": "latency",
        "kinesis": "Low latency, Firehose adds batching delay",
        "kafka": "Very low latency, strong for event streaming",
        "when_to_use": "Kafka for strict streaming/consumer control"
    },
    {
        "dimension": "multi_consumer",
        "kinesis": "Supported, shard read limits matter",
        "kafka": "Strong consumer group model",
        "when_to_use": "Kafka often better for rich multi-team ecosystems"
    },
    {
        "dimension": "pricing",
        "kinesis": "Usage-based AWS pricing",
        "kafka": "Infra/cluster/managed-service pricing",
        "when_to_use": "Depends on scale and cloud posture"
    },
    {
        "dimension": "max_throughput",
        "kinesis": "Scale with shards/on-demand modes",
        "kafka": "Very high with partitioned clusters",
        "when_to_use": "Kafka often wins for very large cross-platform ecosystems"
    },
    {
        "dimension": "when_to_use",
        "kinesis": "AWS-native ingest, Firehose to lake, low ops",
        "kafka": "Cross-platform event backbone, deep replay, broad ecosystem",
        "when_to_use": "Pick based on ops model and ecosystem gravity"
    }
]

headers = ["dimension", "kinesis", "kafka", "when_to_use"]
widths = {h: max(len(h), max(len(str(r[h])) for r in decision_rows)) for h in headers}

def fmt_row(row):
    return " | ".join(str(row[h]).ljust(widths[h]) for h in headers)

separator = "-+-".join("-" * widths[h] for h in headers)

print(fmt_row({h: h for h in headers}))
print(separator)
for row in decision_rows:
    print(fmt_row(row))


## What Just Happened

- We connected to AWS using the **study** profile in **us-east-1**
- We pulled **50 alert records** from PostgreSQL
- We pushed them to **Kinesis Data Streams**
- We read them back from the stream
- We sent the same records into **Kinesis Firehose**
- We confirmed delivery into **S3**
- We sized shard demand for a **Citi-scale** event stream

### Bottom line

**Kinesis is the AWS-native Kafka.**  
For pure AWS shops, **Firehose → S3 → Athena** is the zero-ops streaming pipeline.  
In enterprise patterns, Citi can use Kinesis for telemetry and security event movement, including **CloudTrail-style event streaming into the security lake**.
